In [ ]:
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.optimize import minimize, approx_fprime
import matplotlib.pyplot as plt

# ------------------ CRPS Posterior Core ------------------ #
def crps_log_posterior(mu, sigma, y, w):
    if sigma <= 0:
        return -np.inf
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    log_prior = -np.log(sigma)  # Reference prior
    log_like = -np.sum(crps) / w
    return log_prior + log_like

def metropolis_crps(y, w, n_samples=5000, burnin=1000, proposal_scale_mu=0.03, proposal_scale_sigma=0.05, random_seed=42):
    np.random.seed(random_seed)
    samples = []
    mu, sigma = np.mean(y), np.std(y)
    logp = crps_log_posterior(mu, sigma, y, w)
    accept = 0
    for t in range(n_samples + burnin):
        mu_prop = np.random.normal(mu, proposal_scale_mu)
        sigma_prop = np.abs(np.random.normal(sigma, proposal_scale_sigma))
        logp_prop = crps_log_posterior(mu_prop, sigma_prop, y, w)
        if np.log(np.random.rand()) < logp_prop - logp:
            mu, sigma, logp = mu_prop, sigma_prop, logp_prop
            accept += (t >= burnin)
        if t >= burnin:
            samples.append([mu, sigma])
    samples = np.array(samples)
    print(f"[MCMC] Acceptance rate: {accept / n_samples:.2f}")
    return samples

# ------------------ Laplace Approximation ------------------ #
def crps_log_posterior_vec(params, y, w):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    log_prior = -log_sigma
    log_like = -np.sum(crps) / w
    return -(log_prior + log_like)  # For minimization

def laplace_approximation(y, w, mu_init, sigma_init):
    init = np.array([mu_init, np.log(sigma_init)])
    res = minimize(crps_log_posterior_vec, init, args=(y, w), method='L-BFGS-B')
    mode = res.x
    mu_mode, log_sigma_mode = mode
    sigma_mode = np.exp(log_sigma_mode)
    print(f"[Laplace] MAP (center): mu={mu_mode:.3f}, sigma={sigma_mode:.3f}")

    # Hessian (finite differences, central diff)
    eps = 1e-4
    def grad(params):
        return approx_fprime(params, lambda p: crps_log_posterior_vec(p, y, w), epsilon=eps)
    def hessian(params):
        n = len(params)
        H = np.zeros((n, n))
        for i in range(n):
            p1 = np.array(params)
            p1[i] += eps
            g1 = grad(p1)
            p2 = np.array(params)
            p2[i] -= eps
            g2 = grad(p2)
            H[:, i] = (g1 - g2) / (2 * eps)
        return H

    H = hessian(mode)
    eigvals = np.linalg.eigvals(H)
    reg = max(1e-6, 1e-1 * np.max(np.abs(eigvals)))
    H_reg = H + reg * np.eye(H.shape[0])
    cov = np.linalg.inv(H_reg)

    # Laplace samples
    n_samp = 5000
    laplace_samples = np.random.multivariate_normal(mode, cov, size=n_samp)
    laplace_mu = laplace_samples[:, 0]
    laplace_sigma = np.exp(laplace_samples[:, 1])

    # Filter bad samples
    mask = np.isfinite(laplace_sigma) & (laplace_sigma > 0) & (laplace_sigma < 100)
    laplace_mu = laplace_mu[mask]
    laplace_sigma = laplace_sigma[mask]

    print("[Laplace] mu mean/std:", np.mean(laplace_mu), np.std(laplace_mu))
    print("[Laplace] sigma mean/std:", np.mean(laplace_sigma), np.std(laplace_sigma))
    print(f"[Laplace] Samples after filtering: {len(laplace_mu)}")
    return laplace_mu, laplace_sigma

# ------------------ Visualization ------------------ #
def plot_mcmc_traces(samples, mu_true, sigma_true):
    fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    axs[0].plot(samples[:, 0], lw=0.6)
    axs[0].axhline(mu_true, color='r', linestyle='--', label='True $\mu$')
    axs[0].set_ylabel(r'$\mu$')
    axs[0].legend()
    axs[1].plot(samples[:, 1], lw=0.6)
    axs[1].axhline(sigma_true, color='r', linestyle='--', label='True $\sigma$')
    axs[1].set_ylabel(r'$\sigma$')
    axs[1].set_xlabel('Iteration')
    axs[1].legend()
    plt.suptitle('MCMC Traces for CRPS-Induced Posterior')
    plt.tight_layout()
    plt.show()

def plot_density_comparison(y, mu_true, sigma_true, mu_post, sigma_post, mu_laplace, sigma_laplace):
    ys = np.linspace(mu_true - 5 * sigma_true, mu_true + 5 * sigma_true, 500)
    plt.hist(y, bins=50, density=True, color='gray', alpha=0.3, label='Data Histogram')
    plt.plot(ys, norm.pdf(ys, mu_true, sigma_true), 'r--', lw=2, label='True Density')
    plt.plot(ys, norm.pdf(ys, mu_post, sigma_post), 'b-', lw=2, label='MCMC Posterior Mean Fit')
    plt.plot(ys, norm.pdf(ys, mu_laplace, sigma_laplace), 'g-.', lw=2, label='Laplace Posterior Mean Fit')
    plt.xlabel('$y$')
    plt.ylabel('Density')
    plt.legend()
    plt.title('Laplace vs MCMC Posterior Mean Fit')
    plt.tight_layout()
    plt.show()

# ------------------ Main Execution ------------------ #
if __name__ == "__main__":
    # Simulation settings
    n = 1000
    mu_true, sigma_true = -1.0, 3.0
    y = np.random.normal(mu_true, sigma_true, n)
    w = 0.5

    # MCMC sampling
    samples = metropolis_crps(y, w, n_samples=5000, burnin=1000)
    mu_post = np.mean(samples[:, 0])
    sigma_post = np.mean(samples[:, 1])

    # Laplace approximation
    laplace_mu_samples, laplace_sigma_samples = laplace_approximation(
        y, w, mu_init=mu_post, sigma_init=sigma_post
    )
    mu_laplace_post = np.mean(laplace_mu_samples)
    sigma_laplace_post = np.mean(laplace_sigma_samples)

    # Plots
    plot_mcmc_traces(samples, mu_true, sigma_true)
    plot_density_comparison(
        y, mu_true, sigma_true,
        mu_post, sigma_post,
        mu_laplace_post, sigma_laplace_post
    )


In [ ]:
import numpy as np
from scipy.stats import norm, laplace, skewnorm
from scipy.optimize import minimize, approx_fprime
import matplotlib.pyplot as plt

# ---------- CRPS Normal Posterior and Inference ---------- #
def crps_log_posterior(mu, sigma, y, w):
    if sigma <= 0:
        return -np.inf
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    log_prior = -np.log(sigma)
    log_like = -np.sum(crps) / w
    return log_prior + log_like

def metropolis_crps(y, w, n_samples=5000, burnin=1000, proposal_scale_mu=0.03, proposal_scale_sigma=0.05, random_seed=42):
    np.random.seed(random_seed)
    samples = []
    mu, sigma = np.mean(y), np.std(y)
    logp = crps_log_posterior(mu, sigma, y, w)
    accept = 0
    for t in range(n_samples + burnin):
        mu_prop = np.random.normal(mu, proposal_scale_mu)
        sigma_prop = np.abs(np.random.normal(sigma, proposal_scale_sigma))
        logp_prop = crps_log_posterior(mu_prop, sigma_prop, y, w)
        if np.log(np.random.rand()) < logp_prop - logp:
            mu, sigma, logp = mu_prop, sigma_prop, logp_prop
            accept += (t >= burnin)
        if t >= burnin:
            samples.append([mu, sigma])
    samples = np.array(samples)
    print(f"[MCMC] Acceptance rate: {accept / n_samples:.2f}")
    return samples

def crps_log_posterior_vec(params, y, w):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    log_prior = -log_sigma
    log_like = -np.sum(crps) / w
    return -(log_prior + log_like)  # For minimization

def laplace_approximation(y, w, mu_init, sigma_init):
    init = np.array([mu_init, np.log(sigma_init)])
    res = minimize(crps_log_posterior_vec, init, args=(y, w), method='L-BFGS-B')
    mode = res.x
    mu_mode, log_sigma_mode = mode
    sigma_mode = np.exp(log_sigma_mode)
    print(f"[Laplace] MAP (center): mu={mu_mode:.3f}, sigma={sigma_mode:.3f}")

    # Hessian (finite differences, central diff)
    eps = 1e-4
    def grad(params):
        return approx_fprime(params, lambda p: crps_log_posterior_vec(p, y, w), epsilon=eps)
    def hessian(params):
        n = len(params)
        H = np.zeros((n, n))
        for i in range(n):
            p1 = np.array(params)
            p1[i] += eps
            g1 = grad(p1)
            p2 = np.array(params)
            p2[i] -= eps
            g2 = grad(p2)
            H[:, i] = (g1 - g2) / (2 * eps)
        return H

    H = hessian(mode)
    eigvals = np.linalg.eigvals(H)
    reg = max(1e-6, 1e-1 * np.max(np.abs(eigvals)))
    H_reg = H + reg * np.eye(H.shape[0])
    cov = np.linalg.inv(H_reg)

    n_samp = 5000
    laplace_samples = np.random.multivariate_normal(mode, cov, size=n_samp)
    laplace_mu = laplace_samples[:, 0]
    laplace_sigma = np.exp(laplace_samples[:, 1])
    mask = np.isfinite(laplace_sigma) & (laplace_sigma > 0) & (laplace_sigma < 100)
    laplace_mu = laplace_mu[mask]
    laplace_sigma = laplace_sigma[mask]
    return laplace_mu, laplace_sigma

def plot_all(y, mu_true, sigma_true, samples, mu_post, sigma_post, mu_laplace, sigma_laplace, dist_label):
    ys = np.linspace(np.min(y)-2, np.max(y)+2, 500)
    plt.figure(figsize=(8,5))
    plt.hist(y, bins=50, density=True, color='gray', alpha=0.3, label='Data Histogram')
    # True density for the *data generating* distribution
    if dist_label == 'Laplace':
        plt.plot(ys, laplace.pdf(ys, loc=mu_true, scale=sigma_true/np.sqrt(2)), 'r--', lw=2, label='True Laplace')
    elif dist_label == 'Skew-Normal':
        plt.plot(ys, skewnorm.pdf(ys, a=5, loc=mu_true, scale=sigma_true), 'r--', lw=2, label='True Skew-Normal')
    else:
        plt.plot(ys, norm.pdf(ys, mu_true, sigma_true), 'r--', lw=2, label='True Normal')
    plt.plot(ys, norm.pdf(ys, mu_post, sigma_post), 'b-', lw=2, label='MCMC Posterior Mean (Normal)')
    plt.plot(ys, norm.pdf(ys, mu_laplace, sigma_laplace), 'g-.', lw=2, label='Laplace Posterior Mean (Normal)')
    plt.xlabel('$y$')
    plt.ylabel('Density')
    plt.legend()
    plt.title(f'CRPS-Normal Posterior Fit ({dist_label} Data)')
    plt.tight_layout()
    plt.show()

    # Trace plots
    fig, axs = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
    axs[0].plot(samples[:, 0], lw=0.6)
    axs[0].set_ylabel(r'$\mu$')
    axs[1].plot(samples[:, 1], lw=0.6)
    axs[1].set_ylabel(r'$\sigma$')
    axs[1].set_xlabel('Iteration')
    plt.suptitle(f'MCMC Traces for CRPS-Induced Posterior ({dist_label} Data)')
    plt.tight_layout()
    plt.show()

# -------------- MAIN for two types of data -------------- #
if __name__ == "__main__":
    # Simulation settings
    n = 1000
    w = 0.5

    # --- 1. Laplace data ---
    mu_lap, sigma_lap = 0.0, 2.0
    y_laplace = laplace.rvs(loc=mu_lap, scale=sigma_lap/np.sqrt(2), size=n)  # scale so var matches normal
    samples_lap = metropolis_crps(y_laplace, w, n_samples=4000, burnin=1000)
    mu_post_lap = np.mean(samples_lap[:, 0])
    sigma_post_lap = np.mean(samples_lap[:, 1])
    mu_laplace_samp, sigma_laplace_samp = laplace_approximation(y_laplace, w, mu_init=mu_post_lap, sigma_init=sigma_post_lap)
    plot_all(y_laplace, mu_lap, sigma_lap, samples_lap, mu_post_lap, sigma_post_lap, np.mean(mu_laplace_samp), np.mean(sigma_laplace_samp), "Laplace")

    # --- 2. Skew-normal data ---
    mu_skew, sigma_skew = 1.0, 2.0
    alpha = 5  # skewness parameter: positive = right-skew
    y_skew = skewnorm.rvs(a=alpha, loc=mu_skew, scale=sigma_skew, size=n)
    samples_skew = metropolis_crps(y_skew, w, n_samples=4000, burnin=1000)
    mu_post_skew = np.mean(samples_skew[:, 0])
    sigma_post_skew = np.mean(samples_skew[:, 1])
    mu_laplace_samp2, sigma_laplace_samp2 = laplace_approximation(y_skew, w, mu_init=mu_post_skew, sigma_init=sigma_post_skew)
    plot_all(y_skew, mu_skew, sigma_skew, samples_skew, mu_post_skew, sigma_post_skew, np.mean(mu_laplace_samp2), np.mean(sigma_laplace_samp2), "Skew-Normal")
